# Food Delivery Business Intelligence & Operations Analytics Platform

## Project Overview

This project analyzes operational and business performance data from a food delivery platform. The goal is to understand how the business is performing, identify operational challenges, and uncover opportunities to improve revenue, customer satisfaction, and delivery efficiency.

By examining order transactions, customer behavior, restaurant performance, and delivery operations, this analysis provides practical insights that can help decision-makers improve both profitability and user experience.

---

## Business Context

Food delivery platforms operate in a highly competitive environment where customer expectations are high and profit margins are often tight. Small inefficiencies—such as excessive discounts, delayed food preparation, restaurant cancellations, or delivery delays—can significantly impact revenue and customer retention.

To make informed decisions, stakeholders need visibility into what is happening across the platform and which areas require attention.

---

## Project Objectives

The analysis focuses on answering five key business questions:

### 1. How healthy is the platform's revenue performance?

* Revenue trends
* Average Order Value (AOV)
* Impact of discounts on revenue

### 2. Where are the major operational bottlenecks?

* Kitchen Preparation Time (KPT)
* Rider wait time
* Order cancellations and fulfillment issues

### 3. Which restaurants perform best and worst?

* Revenue contribution
* Customer ratings
* Rejection and cancellation behavior

### 4. What customer behaviors drive business performance?

* Repeat vs. one-time customers
* Customer Lifetime Value (LTV)
* Order frequency patterns

### 5. What actions can the business take to improve performance?

* Operational recommendations
* Customer retention opportunities
* Restaurant partnership improvements

---

## Key Stakeholders

| Stakeholder                      | Key Questions                                                                        |
| -------------------------------- | ------------------------------------------------------------------------------------ |
| **CEO / Business Leadership**    | Is the business growing profitably?                                                  |
| **Operations Team**              | Where are delivery delays and cancellations occurring?                               |
| **Restaurant Partnerships Team** | Which restaurants need support or intervention?                                      |
| **Marketing Team**               | Which customers drive the most value and how effective are discounts?                |
| **Finance Team**                 | How much revenue is lost through discounts, refunds, and operational inefficiencies? |

---

## Dataset Overview

The dataset contains individual food delivery orders placed on the platform.

**Unit of Analysis:** One row represents one customer order.

### Key Dimensions

* Customer
* Restaurant
* Subzone
* City
* Date and Time

### Key Metrics

* Order Value
* Revenue
* Discounts
* Customer Ratings
* Kitchen Preparation Time (KPT)
* Rider Wait Time
* Delivery Duration
* Order Cancellations
* Customer Complaints

The analysis combines these metrics to evaluate business performance from financial, operational, restaurant, and customer perspectives.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Plot style
sns.set_theme(style="whitegrid")

# Load data
df = pd.read_csv("../data/food_delievery_clean.csv")

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

## Section 1 — Data Understanding

### 1.1 Dataset Shape & Column Index

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\nData Types")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col} = {df[col].dtypes}")


### 1.2 Data Types

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
summary=df[numeric_cols].describe().T

summary['median'] = df[numeric_cols].median()
summary['skew'] = df[numeric_cols].skew().round(3)
summary


### 1.3 Platform Scope — Date Range, Scale Metrics

In [ ]:
df["order_placed_at"]= pd.to_datetime(df["order_placed_at"])
date_min = df['order_placed_at'].min()
date_max = df['order_placed_at'].max()
date_range_days = (date_max - date_min).days

unique_customers   = df['customer_id'].nunique()
unique_restaurants = df['restaurant_id'].nunique()
unique_cities      = df['city'].nunique()
unique_subzones    = df['subzone'].nunique()
total_orders       = df['order_id'].nunique()


print("PLATFORM SCOPE SUMMARY")
print(f"Date Range      : {date_min.date()} to {date_max.date()}")
print(f"Span (days)     : {date_range_days}")
print(f"Total Orders    : {total_orders}")
print(f"Unique Customers: {unique_customers}")
print(f"Restaurants     : {unique_restaurants}")
print(f"Cities          : {unique_cities}")
print(f"Subzones        : {unique_subzones}")



## Section 2 — Executive KPI Overview

These are the seven metrics a business head reviews first. Every subsequent section explains the *why* behind these numbers.

In [ ]:
(df.head())


In [ ]:
print(df.columns)
df["kpt_duration_minutes"].value_counts()

In [ ]:
# ── KPI Calculations ──
total_revenue     = df['total'].sum()
total_orders      = len(df)
unique_customers  = df['customer_id'].nunique()
unique_restaurant = df['restaurant_id'].nunique()
aov               = df['total'].mean()
avg_rating        = df['rating'].mean()
avg_RiderWaitTime = df["rider_wait_time_minutes"].mean()
avg_Distance      = df["distance"].mean()

Not_delievered = ['Return cancelled', 'Returned', 'Timed Out', 'Rejected']
Not_delievered_rate = (
    df['order_status'].isin(Not_delievered).sum() / total_orders * 100
)
delievery_success_rate = 100 - Not_delievered_rate 

print("  EXECUTIVE KPI DASHBOARD")
print(f"  Total Revenue             : ₹{total_revenue:.2f}")
print(f"  Total Orders              : {total_orders}")
print(f"  Unique Customers          : {unique_customers}")
print(f"  Unique Restaurants        : {unique_restaurant}")
print(f"  Avg Order Value           : ₹{aov:.2f}")
print(f"  Avg Rating                : {avg_rating:.2f}")
print(f"  Cancellation Rate         : {Not_delievered_rate:.2f}%")
print(f"  Avg Rider Wait Time       : {avg_RiderWaitTime:.2f}")
print(f"  Avg Delievery Distance    : {avg_Distance:.2f}")
print(f"  Delievery success rate    : {delievery_success_rate:.2f}%")


## Section 3 — Revenue Analytics
### 3.1 Revenue Trend Analysis

In [ ]:
# Daily and monthly revenue series
df['order_date'] = df['order_placed_at'].dt.date
df['year_month'] = df['order_placed_at'].dt.to_period('M')

daily_revenue = df.groupby('order_date')['total'].sum()
monthly_revenue = df.groupby('year_month')['total'].sum()

fig, ax = plt.subplots(2, 1, figsize=(12, 8))

# Daily revenue trend
ax[0].plot(daily_revenue.index, daily_revenue.values)
ax[0].plot(
    daily_revenue.index,
    daily_revenue.rolling(7).mean(),
    linewidth=2,
    label='7-Day Average'
)

ax[0].set_title('Daily Revenue Trend')
ax[0].set_ylabel('Revenue (₹)')
ax[0].legend()

# Monthly revenue
ax[1].bar(
    monthly_revenue.index.astype(str),
    monthly_revenue.values
)

ax[1].set_title('Monthly Revenue')
ax[1].set_ylabel('Revenue (₹)')
ax[1].set_xlabel('Month')

plt.tight_layout()
plt.show()

### 3.2 Revenue by Subzone (City is the same)

In [ ]:
# Revenue by subzone
subzone_revenue = (
    df.groupby('subzone')
      .agg(
          revenue=('total', 'sum'),
          orders=('order_id', 'count'),
          aov=('total', 'mean')
      )
      .sort_values('revenue', ascending=False)
)

top20 = subzone_revenue.head(20)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Top revenue-generating subzones
# Top 20 subzones by revenue
sns.barplot(
    data=top20,
    x='revenue',
    y='subzone',
    ax=ax[0]
)

ax[0].set_title('Top 20 Subzones by Revenue')

# Revenue vs Orders
sns.scatterplot(
    data=top20,
    x='orders',
    y='revenue',
    ax=ax[1]
)
ax[1].set_title('Revenue vs Orders')

for zone in top20.index:
    ax[1].annotate(
        zone,
        (top20.loc[zone, 'orders'],
         top20.loc[zone, 'revenue']),
        fontsize=9
    )

plt.tight_layout()
plt.show()

### 3.4 Top Restaurants by Revenue

In [ ]:
# Restaurant revenue summary
rest_revenue = (
    df.groupby(['restaurant_id', 'restaurant_name'])
      .agg(
          total_revenue=('total', 'sum'),
          total_orders=('order_id', 'count')
      )
      .reset_index()
      .sort_values('total_revenue', ascending=False)
)

top15 = rest_revenue.head(15)

bottom10 = (
    rest_revenue[rest_revenue['total_orders'] >= 5]
    .sort_values('total_revenue')
    .head(10)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 restaurants
sns.barplot(
    data=top15,
    x='total_revenue',
    y='restaurant_name',
    ax=axes[0]
)

axes[0].set_title('Top 15 Restaurants by Revenue')
axes[0].set_xlabel('Revenue (₹)')
axes[0].set_ylabel('')

# Bottom 10 restaurants
sns.barplot(
    data=bottom10,
    x='total_revenue',
    y='restaurant_name',
    ax=axes[1]
)

axes[1].set_title('Bottom 10 Restaurants by Revenue')
axes[1].set_xlabel('Revenue (₹)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

### 3.6 Average Order Value (AOV) Analysis

In [ ]:

# Overall AOV distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Order value distribution
sns.histplot(data=df, x='total', bins=50, ax=axes[0])

axes[0].axvline(df['total'].mean(), linestyle='--', label='Mean')
axes[0].axvline(df['total'].median(), linestyle='--', label='Median')
axes[0].set_title('Order Value Distribution')
axes[0].legend()


# Top 15 restaurants by AOV
rest_aov = (
    df.groupby('restaurant_name')
      .filter(lambda x: len(x) >= 10)
      .groupby('restaurant_name')['total']
      .mean()
      .sort_values(ascending=False)
      .head(15)
      .reset_index()
)

sns.barplot(
    data=rest_aov,
    x='total',
    y='restaurant_name',
    ax=axes[1]
)

axes[1].set_title('Top 15 Restaurants by AOV')

plt.tight_layout()
plt.show()

print(f"Overall AOV: ₹{df['total'].mean():.2f}")
print(f"Median Order Value: ₹{df['total'].median():.2f}")


## Section 4 — Order Analytics
### 4.1 - Temporal Demand Pattern (Hour × Day Heatmap)
KPIs:

Orders per day / per hour (peak demand index)
Average items per order
Discount penetration rate (% orders with any discount)
Order fulfillment rate (delivered / total orders placed)
Revenue per order segment (low / mid / high value)

In [ ]:
# Create a pivot table of order counts by day and hour
pivot = df.pivot_table(
    index='day_name',
    columns='hour',
    values='order_id',
    aggfunc='count'
)

# Reorder days correctly
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = pivot.reindex(day_order)

# Plot heatmap
plt.figure(figsize=(14, 5))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, annot=False)
plt.title('Order Volume Heatmap: Hour of Day vs Day of Week')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

# Logic: pivot_table counts how many orders fall in each hour-day combination.
# The heatmap color intensity shows where demand is concentrated.

### 4.2 - Order Value Segmentation (Basket Size Analysis)

In [ ]:
# Bin orders into value segments
df['order_segment'] = pd.cut(
    df['total'],
    bins=[0, 200, 500, float('inf')],
    labels=['Low (<₹200)', 'Mid (₹200–500)', 'High (>₹500)']
)

# Count and percentage per segment
segment_counts = df['order_segment'].value_counts()

# Average rating per segment
segment_rating = df.groupby('order_segment')['rating'].mean()

# Create a comprehensive visualization for order value segmentation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Order count by segment
segment_counts.plot(kind='bar', ax=axes[0])
axes[0].set_title('Order Volume by Segment', fontweight='bold')
axes[0].set_ylabel('Number of Orders')
axes[0].set_xlabel('Order Segment')
axes[0].tick_params(axis='x', rotation=45)

# 2. Average rating by segment
segment_rating.plot(kind='bar', ax=axes[1])
axes[1].set_title('Average Rating by Segment',fontweight='bold')
axes[1].set_ylabel('Average Rating')
axes[1].set_xlabel('Order Segment')
axes[1].set_ylim([4.0, 4.5])
axes[1].tick_params(axis='x', rotation=45)

# Logic: pd.cut() assigns each order to a bucket based on its value.
# You then analyze how behavior differs across those buckets.

### 4.3 Discount penetration rate (% orders with any discount)

In [ ]:
discount_cols = [
    'restaurant_discount_promo',
    'restaurant_discount_flat_offs,_freebies_&_others',
    'gold_discount',
    'brand_pack_discount'
]

# Total cost of each discount type
discount_totals = df[discount_cols].sum().sort_values(ascending=False)

# What % of orders used any discount at all
discount_pct = ((df[discount_cols] > 0).sum() / len(df) * 100).reindex(discount_totals.index)

avg_discount = (
    df[discount_cols]
    .where(df[discount_cols] > 0)
    .mean()
    .reindex(discount_totals.index)
)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

discount_totals.plot.barh(ax=ax[0])
ax[0].set_title('Total Discount Value')

discount_pct.plot.barh(ax=ax[1])
ax[1].set_title('Discount Usage (%)')

plt.tight_layout()
plt.show()

# Logic: sum() across discount columns gives total money given away per type.
# Creating a binary 'has_discount' flag lets you calculate penetration rate simply.

### 4.4. Cancellation Pattern by Time and Day

In [ ]:
# Flag cancelled orders
df['is_cancelled'] = df['order_status'].str.lower().isin(['cancelled', 'rejected'])

# Cancellation rate by hour
cancel_by_hour = df.groupby('hour')['is_cancelled'].mean() * 100

# Cancellation rate by day
cancel_by_day = df.groupby('day_name')['is_cancelled'].mean() * 100

# Reorder days
cancel_by_day = cancel_by_day.reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])

# Plots
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

cancel_by_hour.plot(
    marker='o',
    ax=ax[0],
    title='Cancellation Rate by Hour'
)
ax[0].set_ylabel('Cancellation Rate (%)')

cancel_by_day.plot(
    kind='bar',
    ax=ax[1],
    title='Cancellation Rate by Day'
)
ax[1].set_ylabel('Cancellation Rate (%)')
ax[1].tick_params(rotation=45)

plt.tight_layout()
plt.show()

# Logic: groupby + mean() on a True/False column gives the proportion of
# cancelled orders — which is the cancellation rate — per time group.

### 4.5  Order fulfillment rate = Delivered orders / Total orders placed

## Section 5 — Customer Analytics


### 5.1 Customer Retention — Repeat vs. One-Time Buyers

In [ ]:
# Count orders per customer
orders_per_customer = df.groupby('customer_id')['order_id'].count()

# Label as one-time or repeat
customer_type = orders_per_customer.gt(1).map({True: 'Repeat', False: 'One-Time'})
df['customer_type'] = df['customer_id'].map(customer_type)

# Revenue per customer group
revenue_by_type = df.groupby('customer_type')['total'].sum()

# Plots
fig, ax = plt.subplots(1, 2, figsize=(10,4))

customer_type.value_counts().plot(kind='bar', ax=ax[0])
ax[0].set_title('Customer Distribution')
ax[0].tick_params(rotation=0)

revenue_by_type.plot(kind='bar', ax=ax[1])
ax[1].set_title('Revenue by Customer Type')
ax[1].tick_params(rotation=0)

plt.tight_layout()
plt.show()

# Logic: count orders per customer, then flag >1 as repeat.
# Map that flag back onto the main df to get revenue split.

### 5.2 Customer Order Frequency Distribution

In [ ]:
orders_per_customer = df.groupby('customer_id')['order_id'].count()

# Create frequency buckets
freq_bins = pd.cut(
    orders_per_customer,
    bins=[0, 1, 3, 6, 10, float('inf')],
    labels=['1', '2–3', '4–6', '7–10', '10+']
)
freq_bins.value_counts().sort_index().plot(kind='bar')
plt.title('Orders per Customer Distribution')
plt.xlabel('Number of Orders')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.show()

# Logic: pd.cut() on order counts creates labeled buckets.
# Counting customers per bucket shows the frequency distribution shape.

### 5.3 Customer LTV Distribution & Top Customer Segment

In [ ]:
# Customer Lifetime Value (total spend)
customer_ltv = df.groupby('customer_id')['total'].sum()

# Create 10 customer segments based on spend
deciles = pd.qcut(customer_ltv, 10, labels=range(1, 11))

# Revenue contributed by each decile
decile_revenue = customer_ltv.groupby(deciles).sum()

decile_revenue.plot(
    kind='bar',
    figsize=(8, 4)
)

plt.title('Revenue Contribution by Customer Spend Decile')
plt.xlabel('Customer Decile (1 = Lowest Spend, 10 = Highest Spend)')
plt.ylabel('Revenue (₹)')
plt.xticks(rotation=0)

plt.show()

### 5.4 High-Value vs. Low-Value Customer Behavior Comparison

In [ ]:
# Customer LTV
customer_ltv = (df.groupby('customer_id')['total'].sum()
                .reset_index(name='lifetime_value'))

# Customer segments
customer_ltv['segment'] = pd.qcut(customer_ltv['lifetime_value'],
    q=5,
    labels=['Bottom 20%', 'Q2', 'Q3', 'Q4', 'Top 20%']
)

ax = customer_ltv['segment'].value_counts().sort_index().plot(
    kind='bar',
    figsize=(8,4)
)

ax.set_ylim(2000, 2600)
plt.title('Customer Distribution by LTV Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.show()

customer_ltv.groupby('segment')['lifetime_value'].sum().plot(
    kind='bar',
    figsize=(8,4)
)

plt.title('Revenue Contribution by LTV Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Revenue (₹)')
plt.xticks(rotation=0)

plt.show()

## 6. Restaurant Analysis
### 6.1 Top Restaurants by Composite score 
Business Question: Which restaurants are genuinely high-performing across revenue, rating, speed, and reliability — and which are liabilities?

In [ ]:
# Restaurants with at least 10 orders

rest_score = (
    df.groupby('restaurant_name')
      .agg(
          revenue=('total', 'sum'),
          rating=('rating', 'mean'),
          kpt=('kpt_duration_minutes', 'mean'),
          cancel_rate=('is_cancelled', 'mean')
      )
)

# Normalize metrics
rest_score['revenue_score'] = (rest_score['revenue'] - rest_score['revenue'].min()) / (rest_score['revenue'].max() - rest_score['revenue'].min())

rest_score['rating_score'] = (rest_score['rating'] - rest_score['rating'].min()) / (rest_score['rating'].max() - rest_score['rating'].min())

rest_score['kpt_score'] = 1 - (
    (rest_score['kpt'] - rest_score['kpt'].min()) /
    (rest_score['kpt'].max() - rest_score['kpt'].min())
)

rest_score['cancel_score'] = 1 - (
    (rest_score['cancel_rate'] - rest_score['cancel_rate'].min()) /
    (rest_score['cancel_rate'].max() - rest_score['cancel_rate'].min())
)

# Overall score
rest_score['score'] = rest_score[['revenue_score', 'rating_score', 'kpt_score', 'cancel_score']].mean(axis=1)

rest_score = rest_score.sort_values('score', ascending=False)

# Plot 
# top10 = rest_score.head(10).sort_values("score")

fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(
    rest_score.index,
    rest_score["score"]
)

ax.set_title('Top 10 Restaurants by Performance Score')
ax.set_xlabel('Composite Score')
ax.set_ylabel('Restaurant')

plt.tight_layout()
plt.show()


### 6.2 Restaurant Rejection & Cancellation Analysis
Business Question: Which restaurants have unacceptably high rejection rates.

In [ ]:
# Rejection rate per restaurant
rest_rejection = (
    df.groupby('restaurant_name')
    .agg(
        total_orders=('order_id', 'count'),
        rejections=('is_cancelled', 'sum'),
        avg_penalty=('restaurant_penalty_rejection', 'mean')
    )
)
rest_rejection['rejection_rate'] = (
    rest_rejection['rejections'] / rest_rejection['total_orders'] * 100
)
rest_rejection = rest_rejection.sort_values('rejection_rate', ascending=False)

# Plot 

fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(
    rest_rejection.index,
    rest_rejection['rejection_rate']
)

ax.set_title('Restaurants vs Rejection Rate')
ax.set_xlabel('Rejection Rate (%)')
ax.set_ylabel('Restaurant')

plt.tight_layout()
plt.show()


### 6.3 Restaurant Rating Distribution 
Business Question: What is the rating health of the restaurant catalog?

In [ ]:
rest_ratings = (
    df.groupby('restaurant_name')
    .filter(lambda x: x['rating'].notna().sum() >= 5)
    .groupby('restaurant_name')['rating']
    .mean()
    .reset_index()
)
rest_ratings.columns = ['restaurant_name', 'avg_rating']

# Flag below threshold
# below_threshold = rest_ratings[rest_ratings['avg_rating'] < 4.5].sort_values('avg_rating')

# Plot 
fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(
    rest_ratings['restaurant_name'],
    rest_ratings['avg_rating']
)

ax.set_xlim(3.5, 5.0)
ax.set_xticks([4.0,4.2,4.4,4.6,4.8,5.0])
ax.set_title('Restaurants with Average Ratings')
ax.set_xlabel('Average Rating')
ax.set_ylabel('Restaurant')

plt.tight_layout()
plt.show()

## Section 7 — Operation Analytics
### 7.1 KPT vs. Rider Wait Time — Bottleneck Decomposition
Business Question: Where is delivery time being lost — is it kitchen delay or rider dispatch inefficiency?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    df['kpt_duration_minutes'],
    df['rider_wait_time_minutes']
)

ax.set_title('KPT vs Rider Wait Time')
ax.set_xlabel('Kitchen Preparation Time (minutes)')
ax.set_ylabel('Rider Wait Time (minutes)')

plt.tight_layout()
plt.show()

### 7.2 Delivery Distance vs. Rider Wait Time — Logistics Efficiency
Business Question: Does longer delivery distance drive longer wait times, or is the relationship weaker than expected?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    df['distance'],
    df['rider_wait_time_minutes'],
    alpha=0.3
)

ax.set_title('Distance vs Rider Wait Time')
ax.set_xlabel('Distance (km)')
ax.set_ylabel('Rider Wait Time (minutes)')

plt.tight_layout()
plt.show()

# Correlation
df[['distance', 'rider_wait_time_minutes']].corr()

### 7.3 KPT SLA Breach Analysis
Business Question: What percentage of orders breach a reasonable KPT threshold (e.g., 30 minutes), and which restaurants are the worst offenders?

In [ ]:
# KPT SLA Breach Analysis 

KPT_SLA = 30  # minutes

# Flag SLA breaches
df['kpt_breach'] = df['kpt_duration_minutes'] > KPT_SLA

# Breach rate by restaurant (dataframe)
kpt_breach_rest = (
    df.groupby('restaurant_name')['kpt_breach']
    .mean()
    .mul(100)
    .sort_values()
)

# Plot 
fig, ax = plt.subplots(figsize=(10, 5))

# kpt_breach_rest.plot(
#     kind='barh',
#     ax=ax
# )

ax.barh(
    kpt_breach_rest.index,
    kpt_breach_rest.values

)

ax.set_title('Restaurants by KPT Breach Rate')
ax.set_xlabel('Breach Rate (%)')
ax.set_ylabel('Restaurant')

plt.tight_layout()
plt.show()

### 7.4 Operational Performance by Time of Day
Business Question: Does KPT and rider wait time worsen during peak hours, confirming capacity constraints?

In [ ]:
ops_by_hour = df.groupby('hour').agg(
    avg_kpt=('kpt_duration_minutes', 'mean'),
    avg_rider_wait=('rider_wait_time_minutes', 'mean')
)

fig, ax = plt.subplots(figsize=(10, 5))

ops_by_hour.plot(
    marker='o',
    ax=ax
)

ax.set_title('Average KPT and Rider Wait Time by Hour')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Minutes')

plt.tight_layout()
plt.show()

## Section 8 — Customer Experience Analytics


### 8.1 Customer Complaint Tag Analysis
Business Question: What are customers complaining about most, and how do those complaints map to operational failure types?

In [ ]:

customer_complaints = df[df['customer_complaint_tag'] != 'No Complaint']

complaint_counts = customer_complaints['customer_complaint_tag'].value_counts()

fig, ax = plt.subplots(figsize=(8,4))

complaint_counts.plot(
    kind='barh',
    ax=ax
)

ax.set_title('Complaint Frequency')
ax.set_xlabel('Count')
ax.set_ylabel('Complaint Type')

plt.tight_layout()
plt.show()

### 8.2 Operational Drivers of Rating — KPT & Wait Time vs. Rating
Business Question: Does slower KPT or longer rider wait time directly predict a lower rating?

In [ ]:
# Bin KPT into buckets
df['kpt_bucket'] = pd.cut(
    df['kpt_duration_minutes'],
    bins=[0, 15, 25, 35, float('inf')],
    labels=['0–15 min', '15–25 min', '25–35 min', '35+ min']
)

# Avg rating per KPT bucket
kpt_rating = df.groupby('kpt_bucket')['rating'].mean()
kpt_rating.plot(kind='line', marker='o', color='red', figsize=(9, 4))
plt.title('Avg Rating by KPT Duration Bucket')
plt.ylabel('Avg Rating')
plt.xlabel('KPT Duration')

### 8.3 Cancellation Reason Analysis
Business Question: Why are orders being cancelled, and which cancellation reasons represent fixable operational failures vs. customer behavior?

In [ ]:
# Only cancelled/rejected orders
cancelled_df = df[df['order_status'].str.lower().isin(['cancelled', 'rejected'])]

# Reason frequency
reason_counts = cancelled_df['cancellation___rejection_reason'].value_counts()
reason_counts.plot(kind='barh', figsize=(10, 6))
